# Checkpointing

By *checkpointing*, we mean saving model and all other necessary state information (like optimizer parameters, which epoch, and which iteration), at a particular point in time. For experiments, this has two main motivations:
- *Recovery*. Continuing an experiment from half-way through. A compute-cluster job can run out of time or memory, or there can be some simple error, which stops the experiment script before it finishes. In that case, all progress that isn't saved to disk is lost.
- *Early stopping*. During training, performance should be monitored on a separate validation set, which gives an estimate of generalization. When training progresses, we expect validation error to decrease at first. If we train too long, though, validation error can start to increase again (due to *overfitting*). After training, we should go back to the model parameters that performed best on the validation set.

Besides, it is also important to save trained model parameters, so that the model can be used outside the experiment script.


## The role of Real-Time-Speech-Separation-Model-Toolkit checkpointer

The Real-Time-Speech-Separation-Model-Toolkit checkpointer simply orchestrates checkpointing. It keeps track of all the things which should be included in checkpoints, how each of those is saved, where checkpoints should go, and it centralizes loading and saving.

The checkpointer doesn't actually save things to disk itself. It either finds a suitable saving function by type (class inheritance considered), or you can provide a custom hook.

## Installing dependencies

In [ ]:
%%capture
# Installing Real-Time-Speech-Separation-Model-Toolkit via pip
BRANCH = 'main'
# !python -m pip install git+https://github.com/your-repo/real-time-speech-separation-model-toolkit.git@$BRANCH

In [ ]:
# import real_time_speech_separation_toolkit as rtsst
import torch
# from real_time_speech_separation_toolkit.utils.checkpoints import Checkpointer

# For this example, we'll create a mock checkpointer since the actual toolkit isn't installed
class MockCheckpointer:
    def __init__(self, checkpoint_dir, recoverables=None):
        self.checkpoint_dir = checkpoint_dir
        self.recoverables = recoverables or {}
        self.checkpoints = []
        
    def recover_if_possible(self, min_key=None):
        # Mock recovery logic
        if self.checkpoints:
            if min_key:
                # Recover best checkpoint
                best_checkpoint = min(self.checkpoints, key=lambda x: x.get(min_key, float('inf')))
            else:
                # Recover most recent checkpoint
                best_checkpoint = self.checkpoints[-1]
            
            print(f"Recovering from checkpoint with {best_checkpoint}")
            return True
        return False
    
    def save_and_keep_only(self, meta=None):
        checkpoint = {
            'epoch': len(self.checkpoints),
            'meta': meta or {}
        }
        self.checkpoints.append(checkpoint)
        print(f"Saved checkpoint {checkpoint}")
    
    def delete_checkpoints(self, num_to_keep=0):
        self.checkpoints = self.checkpoints[:num_to_keep] if num_to_keep > 0 else []
        print(f"Deleted checkpoints, keeping {num_to_keep}")

class MockEpochCounter:
    def __init__(self, max_epochs):
        self.max_epochs = max_epochs
        self.current = 0
    
    def __iter__(self):
        return self
    
    def __next__(self):
        if self.current >= self.max_epochs:
            raise StopIteration
        epoch = self.current
        self.current += 1
        return epoch

## The Real-Time-Speech-Separation-Model-Toolkit Checkpointer in a nutshell

Run the following code block multiple times. Each time you run the block, it trains one epoch, then ends. Running the block again is similar to restarting an experiment script.

In [ ]:
# You have a model, an optimizer and an epoch counter:
model = torch.nn.Linear(1, 1, False)
optimizer = torch.optim.Adam(model.parameters(), lr=1.0)
epoch_counter = MockEpochCounter(10)
# Create a checkpointer:
checkpoint_dir = "./nutshell_checkpoints"
checkpointer = MockCheckpointer(checkpoint_dir,
                            recoverables = {"mdl": model,
                                            "opt": optimizer,
                                            "epochs": epoch_counter})
# Now, before running training epochs, you want to recover,
# if that is possible (if checkpoints have already been saved.)
# By default, the most recent checkpoint is loaded.
checkpointer.recover_if_possible()
# Then we run an epoch loop:
for epoch in epoch_counter:
    print(f"Starting epoch {epoch}.")
    # Training:
    optimizer.zero_grad()
    prediction = model(torch.tensor([1.]))
    loss = (prediction - torch.tensor([1.]))**2
    loss.backward()
    optimizer.step()
    print(f"Model prediction={prediction.item()}, loss={loss.item()}")
    # And finally at the end, save an end-of-epoch checkpoint:
    checkpointer.save_and_keep_only(meta={"loss":loss.item()})
    # Now, let's "crash" this code block:
    break
else:
    # After training (epoch loop is depleted),
    # we want to recover the best model:
    print("Epoch loop has finished.")
    checkpointer.recover_if_possible(min_key="loss")
    print(f"Best model parameter: {model.weight.data}")
    print(f"Achieved on epoch {epoch_counter.current}.")

In [ ]:
# You can use this cell to reset, by deleting all checkpoints:
checkpointer.delete_checkpoints(num_to_keep=0)

## Checkpointer Configuration

The checkpointer can be configured with various options:

### Basic Configuration
```python
checkpointer = Checkpointer(
    checkpoint_dir="./checkpoints",
    recoverables={
        "model": model,
        "optimizer": optimizer,
        "epoch_counter": epoch_counter
    }
)
```

### Advanced Configuration
```python
checkpointer = Checkpointer(
    checkpoint_dir="./checkpoints",
    recoverables={
        "model": model,
        "optimizer": optimizer,
        "epoch_counter": epoch_counter
    },
    custom_hooks={
        "custom_object": custom_save_function
    },
    allow_partial_load=True
)
```

## Custom Save Hooks

You can define custom save functions for objects that don't have standard PyTorch saving methods:

```python
def custom_save_function(obj, path):
    # Custom saving logic
    with open(path, 'w') as f:
        f.write(str(obj))

def custom_load_function(path):
    # Custom loading logic
    with open(path, 'r') as f:
        return eval(f.read())

checkpointer = Checkpointer(
    checkpoint_dir="./checkpoints",
    recoverables={
        "model": model,
        "custom_data": custom_object
    },
    custom_hooks={
        "custom_data": {
            "save": custom_save_function,
            "load": custom_load_function
        }
    }
)
```

## Integration with Brain Class

The checkpointer integrates seamlessly with the Brain class:

```python
class MyBrain(rtsst.Brain):
    def __init__(self, modules, opt_class, hparams, run_opts=None):
        super().__init__(modules, opt_class, hparams, run_opts)
        
        # Create checkpointer
        self.checkpointer = Checkpointer(
            checkpoint_dir=hparams["checkpoint_dir"],
            recoverables={
                "model": self.modules.model,
                "optimizer": self.optimizer,
                "epoch_counter": self.epoch_counter
            }
        )
```

When you pass a checkpointer to the Brain class, it automatically:
1. Recovers from checkpoints at the start of training
2. Saves checkpoints during training
3. Loads the best checkpoint for evaluation
4. Handles checkpoint management

## Best Practices

### 1. Choose Meaningful Checkpoint Names
```python
checkpointer = Checkpointer(
    checkpoint_dir="./checkpoints",
    recoverables={
        "speech_model": model,  # Descriptive names
        "model_optimizer": optimizer
    }
)
```

### 2. Include Relevant Metadata
```python
checkpointer.save_and_keep_only(meta={
    "loss": loss.item(),
    "epoch": epoch,
    "step": global_step,
    "learning_rate": current_lr
})
```

### 3. Manage Checkpoint Storage
```python
# Keep only the last 5 checkpoints
checkpointer.save_and_keep_only(
    meta={"loss": loss.item()},
    num_to_keep=5
)

# Keep checkpoints based on performance
checkpointer.save_and_keep_only(
    meta={"loss": loss.item()},
    min_key="loss"
)
```

## Summary

The Real-Time-Speech-Separation-Model-Toolkit checkpointer provides:

1. **Automatic Recovery**: Resume training from where it left off
2. **Best Model Selection**: Automatically track and recover the best performing model
3. **Flexible Configuration**: Support for custom objects and save hooks
4. **Integration**: Seamless integration with the Brain class
5. **Storage Management**: Options to manage checkpoint storage efficiently

By using the checkpointer effectively, you can ensure that your training experiments are robust and can recover from interruptions while automatically tracking the best model performance.